# MAE consolidado

**Lê:** `metrics_*.json` de cada modelo já rodado.

**Grava:** `mae_consolidado.csv`

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

# sobe até achar analyses/grupo2
here = Path.cwd().resolve()
group = None
for c in [here, *here.parents]:
    if (c / 'config.py').exists() and (c / 'lib').is_dir():
        group = c
        break
if group is None:
    raise RuntimeError('Abra/rode este notebook de dentro de analyses/grupo2/...')
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib
from lib import features, eda_stl, models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print('grupo2 →', group)
print('outputs →', config.OUTPUT_DIR)


In [ ]:
rows = []
for model in ['Random_Forest', 'PLS_Regression', 'SARIMAX', 'Holt_Winters']:
    path = config.RESULT_DIR / f'metrics_{model}.json'
    if not path.exists():
        print('ainda falta:', path.name, '→ rode o walk-forward desse modelo')
        continue
    m = load_json(path)
    rows.append({'model': model, 'mae': m['mae'], 'runtime_sec': m['runtime_sec'],
                 'ljung_box_pvalue_lag24': m['ljung_box_pvalue_lag24']})
mae_df = pd.DataFrame(rows).sort_values('mae')
mae_df['rank'] = range(1, len(mae_df) + 1)
mae_df.to_csv(config.RESULT_DIR / 'mae_consolidado.csv', index=False)
display(mae_df)
if len(mae_df):
    print('Melhor:', mae_df.iloc[0]['model'])
